In [6]:
# SECTION: IMPORT LIBRARIES
# Role: Load packages needed for data handling, the feature-based
#       Random Forest model, evaluation and plotting.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Libraries loaded.")

Libraries loaded.


In [7]:
 # SECTION: DATA LOADING
# Role: Read the leaf disease CSV (image paths + labels) into a
#       pandas DataFrame and show its shape/columns.
CSV_PATH = r"C:\Users\user\OneDrive\Desktop\kaggle\leaf_disease_dataset.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"CSV not found at: {CSV_PATH}\n"
        "Double-check the path (copy it from File Explorer's address bar) and update CSV_PATH above."
    )

df = pd.read_csv(CSV_PATH)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head()

Shape: (600, 9)

Columns: ['Leaf_Area', 'Mean_R', 'Mean_G', 'Mean_B', 'Texture', 'Moisture', 'Crop', 'Disease_Spots', 'Disease']


,Leaf_Area,Mean_R,Mean_G,Mean_B,Texture,Moisture,Crop,Disease_Spots,Disease
0,2512.33,118.25,122.59,78.75,0.670,49.58,Potato,9.82,Early_Blight
1,2586.59,135.51,132.67,64.56,0.602,49.16,Tomato,10.82,Early_Blight
2,1904.36,98.79,92.13,72.05,0.793,47.61,Tomato,27.56,Late_Blight
3,2426.93,113.36,131.44,79.24,0.386,53.17,Potato,1.20,Healthy
4,2028.83,116.17,129.83,79.82,0.669,52.45,Potato,11.05,Early_Blight


In [8]:
#First 5 rows

print(df.head())

   Leaf_Area  Mean_R  Mean_G  Mean_B  Texture  Moisture    Crop  \
0    2512.33  118.25  122.59   78.75    0.670     49.58  Potato   
1    2586.59  135.51  132.67   64.56    0.602     49.16  Tomato   
2    1904.36   98.79   92.13   72.05    0.793     47.61  Tomato   
3    2426.93  113.36  131.44   79.24    0.386     53.17  Potato   
4    2028.83  116.17  129.83   79.82    0.669     52.45  Potato   

   Disease_Spots       Disease  
0           9.82  Early_Blight  
1          10.82  Early_Blight  
2          27.56   Late_Blight  
3           1.20       Healthy  
4          11.05  Early_Blight  


In [9]:
#Last 5 rows

print(df.tail())

     Leaf_Area  Mean_R  Mean_G  Mean_B  Texture  Moisture    Crop  \
595    2542.90   96.24  128.41   82.41    0.415     59.61  Potato   
596    2112.55  125.31  121.21   63.47    0.635     54.47  Potato   
597    2274.81  100.08   89.76   61.01    0.774     49.78  Potato   
598    1685.02  138.91  110.96   73.83    0.716     50.77  Tomato   
599    2288.71  138.24  116.37   71.35    0.591     58.66  Potato   

     Disease_Spots       Disease  
595           2.56       Healthy  
596          10.17  Early_Blight  
597          18.96   Late_Blight  
598           9.87  Early_Blight  
599          12.92  Early_Blight  


In [10]:
#Statistical information
#This gives:
# mean
# standard deviation
# minimum
# maximum
# quartiles

print(df.describe())

         Leaf_Area      Mean_R      Mean_G      Mean_B     Texture  \
count   600.000000  600.000000  600.000000  600.000000  600.000000   
mean   2355.061317  109.926950  117.963650   69.303033    0.565310   
std     374.354300   16.451656   23.434427   10.058695    0.213096   
min    1234.970000   60.750000   59.940000   42.630000    0.143000   
25%    2082.000000   99.370000   98.817500   62.815000    0.339750   
50%    2358.460000  108.400000  117.340000   70.185000    0.614000   
75%    2646.605000  121.345000  135.885000   76.090000    0.745000   
max    3484.730000  157.320000  182.320000   95.590000    1.000000   

         Moisture  Disease_Spots  
count  600.000000     600.000000  
mean    59.622600      11.525667  
std      8.564532       8.016274  
min     33.950000       0.000000  
25%     53.530000       2.615000  
50%     59.215000      12.230000  
75%     65.852500      18.270000  
max     83.720000      32.970000  


In [11]:
#Dataset information

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Leaf_Area      600 non-null    float64
 1   Mean_R         600 non-null    float64
 2   Mean_G         600 non-null    float64
 3   Mean_B         600 non-null    float64
 4   Texture        600 non-null    float64
 5   Moisture       600 non-null    float64
 6   Crop           600 non-null    str    
 7   Disease_Spots  600 non-null    float64
 8   Disease        600 non-null    str    
dtypes: float64(7), str(2)
memory usage: 51.7 KB
None


In [12]:
#Data types

print(df.dtypes)

Leaf_Area        float64
Mean_R           float64
Mean_G           float64
Mean_B           float64
Texture          float64
Moisture         float64
Crop                 str
Disease_Spots    float64
Disease              str
dtype: object


In [13]:
#Missing values

print(df.isnull().sum())

Leaf_Area        0
Mean_R           0
Mean_G           0
Mean_B           0
Texture          0
Moisture         0
Crop             0
Disease_Spots    0
Disease          0
dtype: int64


In [14]:
#Find Duplicates

print("Duplicates:", df.duplicated().sum())

Duplicates: 0


In [15]:
#Check for impossible values

print(df[df["Moisture"] < 0])
print(df[df["Leaf_Area"] <= 0])
print(df[df["Disease_Spots"] < 0])


Empty DataFrame
Columns: [Leaf_Area, Mean_R, Mean_G, Mean_B, Texture, Moisture, Crop, Disease_Spots, Disease]
Index: []
Empty DataFrame
Columns: [Leaf_Area, Mean_R, Mean_G, Mean_B, Texture, Moisture, Crop, Disease_Spots, Disease]
Index: []
Empty DataFrame
Columns: [Leaf_Area, Mean_R, Mean_G, Mean_B, Texture, Moisture, Crop, Disease_Spots, Disease]
Index: []


In [17]:
#DATA CLEANING


import pandas as pd
import numpy as np

# LOAD DATASET

df = pd.read_csv(
    r"C:\Users\user\OneDrive\Desktop\kaggle\leaf_disease_dataset.csv"

print("Original Dataset:")
print(df.head())


# DATASET INFORMATION

print("\nDataset Information:")
print(df.info())

print("\nDataset Shape:")
print(df.shape)


# CHECK MISSING VALUES

print("\nMissing Values:")
print(df.isnull().sum())


# CHECK DUPLICATE ROWS

print("\nDuplicate Rows:")
print(df.duplicated().sum())


# REMOVE DUPLICATES

df = df.drop_duplicates()

print("Duplicates Removed Successfully!")


# CLEAN COLUMN NAMES

df.columns = df.columns.str.strip()

print("\nClean Column Names:")
print(df.columns)


# CLEAN TEXT DATA

for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()


# HANDLE MISSING NUMERICAL VALUES

for column in df.select_dtypes(include=np.number).columns:
    df[column] = df[column].fillna(df[column].median())


# HANDLE MISSING CATEGORICAL VALUES

for column in df.select_dtypes(include="object").columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])


# CHECK MISSING VALUES AFTER CLEANING

print("\nMissing Values After Cleaning:")
print(df.isnull().sum())


# CHECK DATA TYPES

print("\nData Types:")
print(df.dtypes)


# DISPLAY CLEANED DATA

print("\nCleaned Dataset:")
print(df.head())

print("\nFinal Shape:")
print(df.shape)


# SAVE CLEANED DATASET

df.to_csv(
    r"C:\Users\user\OneDrive\Desktop\kaggle\leaf_disease_dataset.csv",
    index=False
)

print("\nCleaned dataset saved successfully!")

SyntaxError: '(' was never closed (292664404.py, line 9)